In [55]:
from pathlib import Path
import pandas as pd
from astropy.time import Time
ROOT = Path.cwd().parents[0]
DF_RAW = ROOT / 'data' / 'raw'

object

In [56]:
col_objects = ['oid', 'meanra', 'meandec', 'ndet', 'firstmjd', 'lastmjd']
col_detections = ['oid', 'candid', 'mjd', 'fid', 'magpsf', 'sigmapsf', 'isdiffpos', 'parent_candid', 'ra', 'dec']
col_probabilities = ['oid', 'classifier_name', 'classifier_version', 'class_name', 'probability', 'ranking']

In [57]:
tmp_objects = pd.read_parquet(DF_RAW / 'objects.parquet', columns=col_objects)
print(tmp_objects.shape)
tmp_objects.head()

(20000, 6)


,oid,meanra,meandec,ndet,firstmjd,lastmjd
0,ZTF17aaaaaak,59.128486,49.427638,1,61229.482338,61229.482338
1,ZTF17aaaaaal,59.899025,50.278091,284,58365.475035,61254.486435
2,ZTF17aaaaaam,60.275151,49.811770,182,58441.450301,60966.401991
3,ZTF17aaaaaan,64.334154,45.594258,588,58343.446620,61256.488137
4,ZTF17aaaaaao,63.989736,45.430465,428,58340.493391,61254.488009


In [58]:
# data type, missing value
pd.concat([tmp_objects.dtypes, tmp_objects.isna().sum()], axis=1)

,0,1
oid,str,0
meanra,float64,0
meandec,float64,0
ndet,int32,0
firstmjd,float64,0
lastmjd,float64,0


In [31]:
# Time; mjd to utc
tmp_objects['first_observed'] = Time(tmp_objects['firstmjd'].values, format='mjd').iso
tmp_objects['last_observed'] = Time(tmp_objects['lastmjd'].values, format='mjd').iso
tmp_objects = tmp_objects.drop(columns=['firstmjd', 'lastmjd'])
tmp_objects.head()

,oid,meanra,meandec,ndet,first_observed,last_observed
0,ZTF17aaaaaak,59.128486,49.427638,1,2026-07-08 11:34:34.003,2026-07-08 11:34:34.003
1,ZTF17aaaaaal,59.899025,50.278091,284,2018-09-04 11:24:02.998,2026-08-02 11:40:28.001
2,ZTF17aaaaaam,60.275151,49.811770,182,2018-11-19 10:48:25.998,2025-10-18 09:38:51.996
3,ZTF17aaaaaan,64.334154,45.594258,588,2018-08-13 10:43:08.003,2026-08-04 11:42:55.002
4,ZTF17aaaaaao,63.989736,45.430465,428,2018-08-10 11:50:29.000,2026-08-02 11:42:44.004


In [51]:
# no bad values for oid
print(sorted(tmp_objects.oid.unique())[0], sorted(tmp_objects.oid.unique())[-1])
len(tmp_objects.oid.unique())

ZTF17aaaaaak ZTF17aaafiob


20000

In [60]:
# ndet range
tmp_objects.ndet.min(), tmp_objects.ndet.max()

(np.int32(1), np.int32(3404))

detection

In [ ]:
tmp_detection = pd.read_parquet(DF_RAW / 'detection' / 'detections_0001.parquet', columns=col_detections)
tmp_probability = pd.read_parquet(DF_RAW / 'probability' / 'probabilities_0001.parquet', columns=col_probabilities)

In [72]:
print(tmp_detection.shape)
tmp_detection.head()

(266935, 10)


,oid,candid,mjd,fid,magpsf,sigmapsf,isdiffpos,parent_candid,ra,dec
0,ZTF17aaaaaas,780139024415010003,58534.139028,1,18.408876,0.135092,-1,805139274415015015,68.578037,49.087102
1,ZTF17aaaaaas,996451014415015006,58750.451019,1,19.438852,0.145746,1,1012449364415015008,68.578156,49.087025
2,ZTF17aaaaaas,1122203914415020005,58876.203912,1,18.550800,0.081796,-1,1142204954415015008,68.578022,49.087117
3,ZTF17aaaaaaw,993493532215010009,58747.493530,1,18.800180,0.097762,-1,1009487112215010011,62.784025,46.340743
4,ZTF17aaaaabd,620487892715015024,58374.487893,2,18.885900,0.092108,1,631506042715010002,71.898601,46.312892


In [78]:
pd.concat([tmp_detection.dtypes, tmp_detection.isna().sum()], axis=1)

,0,1
oid,str,0
candid,int64,0
mjd,float64,0
fid,int32,0
magpsf,float32,0
sigmapsf,float32,0
isdiffpos,int16,0
parent_candid,int64,0
ra,float64,0
dec,float64,0


In [ ]:
for item in Path(DF_RAW / 'detection').glob("*.parquet"): 
    if item.is_file():
        df_tmp = pd.read_parquet(item, columns=col_detections)
        print(f"\n--- {item.name} ---")
        print(pd.concat([df_tmp.dtypes, df_tmp.isna().sum()], axis=1))


--- detections_0001.parquet ---
                     0  1
oid                str  0
candid           int64  0
mjd            float64  0
fid              int32  0
magpsf         float32  0
sigmapsf       float32  0
isdiffpos        int16  0
parent_candid    int64  0
ra             float64  0
dec            float64  0

--- detections_0002.parquet ---
                     0  1
oid                str  0
candid           int64  0
mjd            float64  0
fid              int32  0
magpsf         float32  0
sigmapsf       float32  0
isdiffpos        int16  0
parent_candid    int64  0
ra             float64  0
dec            float64  0

--- detections_0003.parquet ---
                     0  1
oid                str  0
candid           int64  0
mjd            float64  0
fid              int32  0
magpsf         float32  0
sigmapsf       float32  0
isdiffpos        int16  0
parent_candid    int64  0
ra             float64  0
dec            float64  0

--- detections_0004.parquet ---
          

In [114]:
import numpy as np
global_max = -np.inf
global_min = np.inf

for item in Path(DF_RAW / 'detection').glob("*.parquet"):
    if item.is_file():
        df_tmp = pd.read_parquet(item, columns=['magpsf'])

        if len(df_tmp) == 0:
            continue
            
        file_max = df_tmp['magpsf'].max()
        file_min = df_tmp['magpsf'].min()
        
        if pd.notna(file_max) and file_max > global_max:
            global_max = file_max
        if pd.notna(file_min) and file_min < global_min:
            global_min = file_min

print(f"Max magpsf: {global_max}")
print(f"Min magpsf: {global_min}")

Max magpsf: 21.453899383544922
Min magpsf: 10.168000221252441


In [77]:
tmp_detection.fid.unique()

array([1, 2, 3], dtype=int32)

probability

In [73]:
print(tmp_probability.shape)
tmp_probability.head()

(181884, 6)


,oid,classifier_name,classifier_version,class_name,probability,ranking
0,ZTF17aaaaaxa,lc_classifier,hierarchical_rf_1.1.0,E,0.281008,2
1,ZTF17aaaaaxa,lc_classifier,hierarchical_rf_1.1.0,RRL,0.027664,6
2,ZTF17aaaadvl,lc_classifier,hierarchical_rf_1.1.0,LPV,0.009580,7
3,ZTF17aaaaanl,lc_classifier,hierarchical_rf_1.1.0,Blazar,0.003100,10
4,ZTF17aaaaeac,lc_classifier,hierarchical_rf_1.1.0,Blazar,0.104160,3


In [89]:
pd.concat([tmp_probability.dtypes, tmp_probability.isna().sum()], axis=1)

,0,1
oid,str,0
classifier_name,str,0
classifier_version,str,0
class_name,str,0
probability,float32,0
ranking,int16,0


In [111]:
class_names = set()
classifiers = set()

for item in Path(DF_RAW / 'probability').glob("*.parquet"): 
    if item.is_file():
        df_tmp = pd.read_parquet(item, columns=col_probabilities)
        class_names.update(df_tmp.class_name)
        classifiers.update(df_tmp.classifier_name)
        print(f"\n--- {item.name} ---")
        print(pd.concat([df_tmp.dtypes, df_tmp.isna().sum()], axis=1))
        


--- probabilities_0001.parquet ---
                          0  1
oid                     str  0
classifier_name         str  0
classifier_version      str  0
class_name              str  0
probability         float32  0
ranking               int16  0

--- probabilities_0002.parquet ---
                          0  1
oid                     str  0
classifier_name         str  0
classifier_version      str  0
class_name              str  0
probability         float32  0
ranking               int16  0

--- probabilities_0003.parquet ---
                          0  1
oid                     str  0
classifier_name         str  0
classifier_version      str  0
class_name              str  0
probability         float32  0
ranking               int16  0

--- probabilities_0004.parquet ---
                          0  1
oid                     str  0
classifier_name         str  0
classifier_version      str  0
class_name              str  0
probability         float32  0
ranking            

In [98]:
class_names

{'AGN',
 'Blazar',
 'CEP',
 'CV/Nova',
 'DSCT',
 'E',
 'EA',
 'EB/EW',
 'LPV',
 'Microlensing',
 'Periodic',
 'Periodic-Other',
 'QSO',
 'RRL',
 'RRLab',
 'RRLc',
 'RSCVn',
 'SESN',
 'SLSN',
 'SN',
 'SNII',
 'SNIIb',
 'SNIIn',
 'SNIa',
 'SNIbc',
 'Stochastic',
 'TDE',
 'Transient',
 'VS',
 'YSO',
 'asteroid',
 'bogus',
 'satellite'}

In [112]:
classifiers

{'LC_classifier_ATAT_forced_phot(beta)',
 'LC_classifier_BHRF_forced_phot(beta)',
 'lc_classifier',
 'lc_classifier_BHRF_forced_phot',
 'lc_classifier_BHRF_forced_phot_periodic',
 'lc_classifier_BHRF_forced_phot_stochastic',
 'lc_classifier_BHRF_forced_phot_top',
 'lc_classifier_BHRF_forced_phot_transient',
 'lc_classifier_periodic',
 'lc_classifier_stochastic',
 'lc_classifier_top',
 'lc_classifier_transient',
 'stamp_classifier',
 'stamp_classifier_2025_beta'}